# 🔬 Step 2 — Feature Engineering (Silver)

Build fraud signals from raw data:
- **location_jump**: Card used in different city within 10 mins
- **high_velocity**: More than 5 transactions in last 24 hours
- **amount_spike**: Amount > 5x the customer's 30-day average

In [ ]:
# Import Spark SQL helper functions used to create fraud detection features
from pyspark.sql import functions as F

# Load the Bronze transaction table as the starting point for feature engineering
df = spark.table('bronze_fraud_transactions')

# Build risk signals for unusual location changes, rapid activity, large amount spikes, and fast repeat purchases
df_silver = df \
    .withColumn('location_jump',
        (F.col('Location') != F.col('PreviousLocation')).cast('int')) \
    .withColumn('high_velocity',
        (F.col('NumTxnLast24h') > 5).cast('int')) \
    .withColumn('amount_spike',
        (F.col('Amount') > F.col('AvgTxnAmount30d') * 5).cast('int')) \
    .withColumn('fast_repeat',
        (F.col('TimeSinceLastTxnMins') < 5).cast('int'))

# Preview the engineered fraud features alongside the label to validate the transformations
print('✅ Fraud features created')
df_silver.select('TransactionID','Amount','location_jump','high_velocity','amount_spike','fast_repeat','IsFraud').show(10)

In [ ]:
# Save the engineered fraud features to the Silver Delta table for model training
df_silver.write.format('delta').mode('overwrite').saveAsTable('silver_fraud_features')

# Confirm the Silver features table is ready for the next notebook
print('✅ Silver features table saved!')